# Jigsaw Puzzle Reconstruction - Milestone 1 Demonstration**CSE480 Machine Vision** - classical computer vision only.This notebook walks through the six tasks of the brief on a real puzzle,stage by stage, and finishes with the end-to-end routine and itsquantitative evaluation.Everything shown is computed by `src/`, which depends on NumPy alone.Matplotlib is used here only to draw the figures.

In [ ]:
import os, sys, timesys.path.insert(0, os.path.abspath(".."))import numpy as npimport matplotlib.pyplot as pltfrom src import PuzzleSolver, solve_puzzlefrom src import assembly as asmfrom src import contour_extraction as cefrom src import edge_detection as edfrom src import edge_matching as emfrom src import enhancement as enhfrom src import evaluation as evfrom src import piece_description as pdscfrom src import segmentation as segfrom src import thresholding as thplt.rcParams["figure.figsize"] = (12, 6)plt.rcParams["image.cmap"] = "gray"def show(images, titles=None, cols=4, size=3.2):    n = len(images)    rows = (n + cols - 1) // cols    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))    axes = np.atleast_1d(axes).ravel()    for ax, im in zip(axes, images):        a = np.asarray(im)        ax.imshow(a, cmap=None if a.ndim == 3 else "gray")        ax.axis("off")    for ax in axes[n:]:        ax.axis("off")    if titles:        for ax, t in zip(axes, titles):            ax.set_title(t, fontsize=9)    plt.tight_layout(); plt.show()

## 0. The inputTwo kinds of input are used throughout.* A **real photograph** from the provided dataset: 35 pieces of one jigsaw  scattered on a dark cloth.* A **synthetic puzzle** produced by `evaluation.generate_puzzle`, which cuts  any picture into interlocking pieces, shuffles and rotates them, and  remembers the answer. The dataset has no answer key, so this is what makes  reconstruction accuracy measurable at all.

In [ ]:
from PIL import ImageDETECTION = os.path.abspath(os.path.join("..", "detection"))def imread(path, max_side=None):    img = Image.open(path).convert("RGB")    if max_side and max(img.size) > max_side:        s = max_side / max(img.size)        img = img.resize((int(img.size[0] * s), int(img.size[1] * s)), Image.BILINEAR)    return np.asarray(img, dtype=np.uint8)# a real scattered-pieces photograph (falls back to a synthetic scene)photo = Nonelab_dir = os.path.join(DETECTION, "labels", "train")if os.path.isdir(lab_dir):    for name in sorted(os.listdir(lab_dir)):        p = os.path.join(lab_dir, name)        if sum(1 for l in open(p) if l.strip()) >= 35:            photo = imread(os.path.join(DETECTION, "images", "train",                                        name.replace(".txt", ".jpg")), 1280)            print("photograph:", name)            break# a synthetic puzzle with ground truthoriginal = ev.synthetic_source_image(480, 600, seed=114)scrambled, gt = ev.generate_puzzle(original, rows=4, cols=5, rotate=True, seed=14)print("synthetic puzzle:", gt.grid_shape, "pieces, canvas", scrambled.shape)show([photo if photo is not None else scrambled, original, scrambled],     ["dataset photograph", "original picture", "scrambled synthetic puzzle"],     cols=3, size=5)

## 1. Image enhancementNoise reduction (Gaussian and median), contrast adjustment (histogramequalisation and contrast stretching, both driven by the library's ownhistogram) and sharpening (unsharp masking and a Laplacian built on theshared convolution routine).The median filter is the interesting one: the median is a *rank* statistic,not a linear operator, so it has no convolutional or separable form and everyoutput pixel genuinely needs its own window's order statistics. Theimplementation materialises all windows at once through a stride-trick viewso a single vectorised `np.median` does the work in compiled code; the onlyPython loop left is over the colour channels.

In [ ]:
crop = (photo if photo is not None else scrambled)[:360, :480]rng = np.random.default_rng(0)noisy = np.clip(crop / 255.0 + rng.normal(0, 0.05, crop.shape), 0, 1)sp = noisy.copy()sp[rng.random(sp.shape[:2]) < 0.02] = 1.0     # saltsp[rng.random(sp.shape[:2]) < 0.02] = 0.0     # peppershow([crop, sp, enh.gaussian_blur(sp, 1.5), enh.median_filter(sp, 3)],     ["original", "salt and pepper", "Gaussian sigma=1.5", "median 3x3"], cols=4)for name, img in [("no filtering", sp),                  ("gaussian 1.5", enh.gaussian_blur(sp, 1.5)),                  ("median 3", enh.median_filter(sp, 3)),                  ("median 5", enh.median_filter(sp, 5))]:    m = ev.image_metrics(img, crop, allow_rotation=False)    print(f"{name:14s} PSNR {m['psnr_db']:6.2f} dB   SSIM {m['ssim']:.3f}")

In [ ]:
eq = enh.histogram_equalization(crop)st = enh.contrast_stretch(crop, 2, 98)show([crop, eq, st, enh.unsharp_mask(crop, 1.5, 1.2), enh.laplacian_sharpen(crop, 0.35)],     ["original", "histogram equalised", "contrast stretched",      "unsharp mask", "Laplacian sharpen"], cols=5, size=3.0)fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))ax[0].plot(enh.histogram(crop), label="original")ax[0].plot(enh.histogram(eq), label="equalised")ax[0].set_title("histogram"); ax[0].legend()ax[1].plot(enh.cumulative_histogram(enh.histogram(crop)), label="original")ax[1].plot(enh.cumulative_histogram(enh.histogram(eq)), label="equalised")ax[1].set_title("cumulative histogram (the transfer function)"); ax[1].legend()plt.tight_layout(); plt.show()

## 1d. ThresholdingGlobal (fixed and iterative/isodata), Otsu, and adaptive (mean via anintegral image, and Gaussian-weighted).Otsu maximises the between-class variance `w0*w1*(mu0-mu1)^2`. When the twomodes are well separated that criterion is *flat* across the whole emptyvalley between them, so this implementation returns the **midpoint of themaximising plateau** rather than its left edge - the same optimum, but as farfrom both modes as possible.

In [ ]:
gray = enh.gaussian_blur(crop, 1.0)t_otsu, t_iso = th.otsu_threshold(gray), th.isodata_threshold(gray)print(f"Otsu {t_otsu:.3f}   isodata {t_iso:.3f}")show([crop,      th.global_threshold(gray, 0.5),      th.otsu(gray),      th.adaptive_threshold(gray, 51, 0.02, "mean"),      th.adaptive_threshold(gray, 51, 0.02, "gaussian"),      seg.background_distance(crop)],     ["input", "global 0.5", f"Otsu {t_otsu:.2f}", "adaptive mean 51",      "adaptive Gaussian 51", "colour distance to background"], cols=3, size=4)

## 2. Edge detectionSobel and Prewitt return gradient magnitude **and** orientation; Canny addsnon-maximum suppression, double thresholding and hysteresis-based edgelinking.

In [ ]:
_, _, sob_mag, sob_dir = ed.sobel(crop)_, _, pre_mag, _ = ed.prewitt(crop)edges, stages = ed.canny(crop, sigma=1.4, low=0.05, high=0.15, return_stages=True)def orientation_rgb(theta, mag):    a = (np.rad2deg(theta) % 180.0) / 180.0    rgb = np.stack([np.abs(np.sin(np.pi * (a + k / 3))) for k in range(3)], axis=2)    return rgb * mag[:, :, None]show([sob_mag, pre_mag, orientation_rgb(sob_dir, sob_mag),      stages["nms"], stages["strong"], edges],     ["Sobel magnitude", "Prewitt magnitude", "Sobel orientation",      "after non-max suppression", "strong edges", "Canny (after linking)"],     cols=3, size=4)

In [ ]:
# effect of the parameterssigmas = [0.8, 1.4, 2.5]pairs = [(0.02, 0.06), (0.05, 0.15), (0.10, 0.25)]show([ed.canny(crop, sigma=s, low=0.05, high=0.15) for s in sigmas]     + [ed.canny(crop, 1.4, lo, hi) for lo, hi in pairs],     [f"sigma={s}" for s in sigmas] + [f"low={lo}, high={hi}" for lo, hi in pairs],     cols=3, size=4)

## 3. Segmentation and contour extractionForeground mask, then connected components (two-pass union-find over rowruns), then Moore-neighbour boundary tracing, then one cropped `Piece` percomponent with its mask, contour, centroid and normalised orientation.

In [ ]:
scene = photo if photo is not None else scrambledmask = seg.foreground_mask(scene, "background", open_radius=2, close_radius=2)labels_raw, n_raw = seg.connected_components(mask)labels, stats = seg.filter_components(labels_raw, min_area_ratio=0.45)pieces = ce.extract_pieces(scene, labels, stats=stats)print(f"{n_raw} raw components -> {len(stats)} pieces")rng = np.random.default_rng(0)palette = rng.integers(60, 256, size=(labels_raw.max() + 1, 3)); palette[0] = 0show([scene, mask, palette[labels_raw].astype(np.uint8),      palette[labels].astype(np.uint8)],     ["input", "foreground mask", f"{n_raw} raw components",      f"{len(stats)} kept as pieces"], cols=2, size=6)

In [ ]:
# the traced boundary of one piece, and the pieces normalised uprightp = max(pieces, key=lambda q: q.area)outline = np.zeros(p.mask.shape, bool)outline[p.contour[:, 0], p.contour[:, 1]] = Trueshow([p.image, p.mask, outline], ["cropped piece", "its mask",     f"traced contour ({len(p.contour)} points)"], cols=3, size=4)show([ce.normalize_piece(q).image for q in pieces[:8]],     [f"piece {q.index}" for q in pieces[:8]], cols=4, size=2.4)

## 4. Piece-edge descriptionFour corners, then four sides, then each side classified `tab` / `blank` /`flat` from its deviation profile, plus a strip of colour sampled just insideit.Corners come from the **body-edge model**: a piece is a rectangle plus bumps,so its four straight edges share one direction modulo 90 degrees, recoveredas `arg(sum exp(4i*phi))/4` over the tangent angles. A tab arc sweeps everydirection and cancels itself out of that sum.

In [ ]:
res = PuzzleSolver().solve(scrambled, grid_shape=(4, 5))descs = res.descriptionsprint(f"{len(descs)} pieces described; grid used: {res.grid_shape}")colours = {"tab": (0, 0.9, 0), "blank": (0, 0.55, 1.0), "flat": (1.0, 0.85, 0)}fig, axes = plt.subplots(3, 4, figsize=(13, 10))for ax, d in zip(axes.ravel(), descs[:12]):    ax.imshow(d.piece.image)    for s in d.sides:        ax.plot(s.points[:, 1], s.points[:, 0], color=colours[s.type], lw=2)    ax.plot(d.corners[:, 1], d.corners[:, 0], "r.", ms=12)    ax.set_title(f"piece {d.index}: " + "/".join(s.type[0] for s in d.sides), fontsize=9)    ax.axis("off")plt.tight_layout(); plt.show()rows, cols = gt.grid_shapeprint("expected corner/edge/interior pieces:", 4,      2 * (rows - 2) + 2 * (cols - 2), (rows - 2) * (cols - 2))print("observed:",      sum(1 for d in descs if d.n_flats == 2),      sum(1 for d in descs if d.n_flats == 1),      sum(1 for d in descs if d.n_flats == 0))

In [ ]:
# the two descriptors of one piece: shape profile and colour stripd = descs[0]fig, axes = plt.subplots(2, 4, figsize=(14, 4.5))for k, s in enumerate(d.sides):    axes[0, k].plot(s.profile)    axes[0, k].axhline(0, color="k", lw=0.5)    axes[0, k].set_ylim(-0.4, 0.4)    axes[0, k].set_title(f"side {k}: {s.type}", fontsize=9)    axes[1, k].imshow(np.repeat(s.colors.transpose(1, 0, 2), 10, axis=0))    axes[1, k].axis("off")axes[0, 0].set_ylabel("deviation / chord")plt.tight_layout(); plt.show()

## 5. Piece-edge matching$$D(a,b) = w_s D_{shape}(a,b) + w_c D_{colour}(a,b) + w_l D_{length}(a,b)$$with $D=\infty$ for inadmissible pairs (a flat is a puzzle border and cannever be an interior seam; a tab must meet a blank). When two pieces sit sideby side, one side is traversed in the opposite direction to the other andtheir outward normals oppose, so a perfect fit satisfies$p_a(t) = -p_b(1-t)$; the RMS residual of that identity is the shape cost.The colour strips must satisfy $C_a(t) = C_b(1-t)$.

In [ ]:
table = res.tableassoc = ev.associate_with_ground_truth(res.pieces, gt)acc = ev.matching_accuracy(table, descs, gt, assoc)print("weights:", table.weights.as_dict())print(f"true partner ranked first: {acc['top1_accuracy']:.1%}  "      f"(top-3 {acc['top3_accuracy']:.1%}, mean rank {acc['mean_rank']:.1f})")print("best-buddy pairs:", len(em.best_buddies(table)))# separability of the two termsstep = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}opp = {"N": "S", "E": "W", "S": "N", "W": "E"}cell, dirs = {}, {}for i in range(len(descs)):    r, c, ang = gt.placements[assoc[i]]    cell[i] = (int(r), int(c)); dirs[i] = ev.gt_side_directions(descs[i], ang)by_cell = {v: k for k, v in cell.items()}true_s, true_c = [], []for i in range(len(descs)):    r, c = cell[i]    for s in range(4):        dd = dirs[i][s]        j = by_cell.get((r + step[dd][0], c + step[dd][1]))        if j is None:            continue        t = dirs[j].index(opp[dd])        true_s.append(table.shape[i, s, j, t]); true_c.append(table.colour[i, s, j, t])fin = np.isfinite(table.cost)fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))for k, (name, tv, av) in enumerate([("shape", true_s, table.shape[fin]),                                    ("colour", true_c, table.colour[fin])]):    ax[k].hist(av, bins=50, density=True, alpha=0.6, label="all admissible pairs")    ax[k].hist(tv, bins=25, density=True, alpha=0.8, label="true seams")    ax[k].set_title(f"{name}: true {np.mean(tv):.3f} vs all {np.mean(av):.3f}")    ax[k].legend(fontsize=8)plt.tight_layout(); plt.show()

## 6. Assembly and the end-to-end routineGreedy best-first placement on the grid, resolving each piece's **rotation**as it is placed, with best-buddy seams committed first, documentedtie-breaking, three-stage dead-end relaxation, and restarts from everycandidate corner seed so the best arrangement obtained is always returned.

In [ ]:
a = res.assemblypid, rot = a.as_arrays()print("piece grid:")print(pid)print("rotation grid (index of the side facing North):")print(rot)print("complete:", a.complete, " forced placements:", a.n_forced)for line in a.log[:6]:    print(" ", line)show([scrambled, res.reconstruction, original],     ["input (scrambled)", "reconstruction", "original picture"], cols=3, size=5)

In [ ]:
print(res.summary())print()print("reference-free quality")for k, v in res.quality.items():    print(f"  {k:24s} {v}")print()print("against the ground truth")print("  position           ", ev.direct_accuracy(a, descs, gt, assoc))print("  neighbour          ", ev.neighbour_accuracy(a, gt, assoc))print("  rotation           ", ev.rotation_accuracy(a, descs, gt, assoc))print("  image vs original  ", ev.image_metrics(res.reconstructed_body(), original))print()print("timings (seconds)")for k, v in res.timings.items():    print(f"  {k:16s} {v:6.2f}")

## How performance scalesRun time is dominated by scoring all $(4N)^2$ side pairs and by re-runningthe greedy pass from each candidate seed, so it grows roughly with the squareof the piece count. Accuracy is essentially unaffected by rotation - theorientation is resolved during placement - and degrades only when thecompatibility measure itself becomes ambiguous, which happens on pieces whosepicture content is nearly uniform.

In [ ]:
rows = []for (r, c, seed) in [(2, 3, 11), (3, 4, 12), (4, 5, 14), (5, 7, 16)]:    src = ev.synthetic_source_image(110 * r, 110 * c, seed=100 + seed)    scr, g = ev.generate_puzzle(src, rows=r, cols=c, rotate=True, seed=seed)    t0 = time.perf_counter()    out = PuzzleSolver().solve(scr, (r, c))    dt = time.perf_counter() - t0    at = ev.associate_with_ground_truth(out.pieces, g)    rows.append((f"{r}x{c}", r * c, dt,                 ev.neighbour_accuracy(out.assembly, g, at)["neighbour_accuracy"],                 ev.direct_accuracy(out.assembly, out.descriptions, g, at)["position_accuracy"]))    print(f"{rows[-1][0]:>5s}  {r*c:3d} pieces  {dt:6.2f}s  "          f"neighbour {rows[-1][3]:.2f}  position {rows[-1][4]:.2f}")n = [r[1] for r in rows]; t = [r[2] for r in rows]plt.figure(figsize=(5, 3.2))plt.plot(n, t, "o-")plt.xlabel("pieces"); plt.ylabel("seconds"); plt.title("run time vs puzzle size")plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## The real photographsThe dataset photographs are the actual target of this milestone. Two thingsmake them harder than a synthetic cut:* several pieces physically **touch**, and two touching pieces are one  connected component -- no threshold separates them, so a  distance-transform watershed (`segmentation.split_touching`) is needed;* the pieces lie all over a table under uneven light, so colours across a  true seam disagree even where the picture continues -- hence  `colour_norm="meanstd"`.The dataset also carries a hidden answer key: the class ids are the row-majorpositions of the finished 5x7 puzzle, which lets reconstruction be scored onreal photographs and not only on synthetic ones.

In [ ]:
if photo is not None:    out = PuzzleSolver(open_radius=2, close_radius=2, min_area_ratio=0.45,                       max_area_ratio=1.7, colour_norm="meanstd").solve(photo, (5, 7))    print(out.summary())    for note in out.notes:        print(" ", note)    print("flat sides found:",          sum(1 for d in out.descriptions for s in d.sides if s.is_flat), "of 24")    print("corner pieces:", sum(1 for d in out.descriptions if d.is_corner_piece), "of 4")    show([photo, out.mask, out.reconstruction],         ["photograph", "foreground mask", "reconstruction"], cols=3, size=5)else:    print("dataset not available in this checkout")